In [ ]:
# MLP model to predict chirality indices (n, m)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_curve,
    auc,
    f1_score
)

In [ ]:
# configuration
batch_size = 32
num_epochs = 300

seed = 43
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
# loading data
data = pd.read_csv("/content/CNT Select Indices, DataProduction - Select (n,m).csv")

features = ["n", "m"]
target = "diameter (nm)"

# drop rows where the target variable is NaN
data.dropna(subset=[target], inplace=True)

# filter out rows where the target is not 0 or 1
data = data[data[target].isin([0, 1])]

X = data[features].values.astype(np.float32)
y = data[target].values.astype(np.float32)

In [ ]:
# train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
# plotting data distributions
print("\n generating data distribution plots...")

# plot training data distribution
unique_train, counts_train = np.unique(y_train, return_counts=True)

plt.figure(figsize=(8, 6))
labels_train = ['metallic (1)', 'semiconducting (0)']
plt.bar(labels_train, counts_train, color=['blue', 'orange'])

plt.xlabel('metallicity class')
plt.ylabel('count')
plt.title('distribution of metallicity - training data')
plt.savefig('distribution_train.png')
print("saved 'distribution_train.png'")

# plot testing data distribution
unique_test, counts_test = np.unique(y_test, return_counts=True)

plt.figure(figsize=(8, 6))
labels_test = ['metallic (1)', 'semiconducting (0)']
plt.bar(labels_test, counts_test, color=['blue', 'orange'])

plt.xlabel('metallicity class')
plt.ylabel('count')
plt.title('distribution of metallicity - testing data')
plt.savefig('distribution_test.png')
print("saved 'distribution_test.png'")

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),  batch_size=batch_size, shuffle=False)

In [ ]:
# defining a simple MLP
class MLPClassification(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.layers(x)

model = MLPClassification()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# training
print("\nstarting training...")
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0

    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        # reshape labels to be [batch_size, 1] for BCEWithLogitsLoss
        labels = labels.to(device).float()

        outputs = model(feats)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"epoch [{epoch+1}/{num_epochs}], loss = {running_loss/len(train_loader):.4f}")

train_end = time.time()
train_time = train_end - start_time
print(f"\ntraining finished in {train_time:.2f} seconds")

In [ ]:
# testing + metrics
model.eval()
y_pred_list = []
y_true_list = []

with torch.no_grad():
    for bx, by in test_loader:
        logits = model(bx)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        y_pred_list.extend(preds.numpy())
        y_true_list.extend(by.numpy())

y_pred = np.array(y_pred_list).ravel()
y_true = np.array(y_true_list).ravel()

test_end = time.time()
test_time = test_end - train_end
total_time = test_end - start_time

accuracy = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

total_time = time.time() - start_time


In [ ]:
# calculate initial metrics (using default threshold 0.5 from model.predict)
initial_accuracy = accuracy_score(y_test, y_pred)
print(f"\ninitial test accuracy (default threshold 0.5): {initial_accuracy*100:.2f}%")

print("\nclassification report (default threshold 0.5):")
print(classification_report(y_test, y_pred, target_names=["Semi (0)", "Metal (1)"]))

# find best F1 threshold
best_f1 = 0
best_t = 0.5

for t in np.linspace(0.1, 0.9, 81):
    current_preds = (probs > t).astype(int)
    f1 = f1_score(y_test, current_preds)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"\nbest threshold for F1-score: {best_t:.2f}")

# generate final predictions using the best threshold
final_preds = (probs > best_t).astype(int)

# calculate metrics with optimized threshold
optimized_accuracy = accuracy_score(y_test, final_preds)
print(f"\nfinal test accuracy (optimized threshold): {optimized_accuracy*100:.2f}%")

print("\nclassification report (optimized threshold):")
print(classification_report(y_test, final_preds, target_names=["Semi (0)", "Metal (1)"]))

# calculate confusion matrix for plotting
cm = confusion_matrix(y_test, final_preds)

# calculate ROC curve and AUC for plotting
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

# time summary
print("\ntime summary")
print(f"training time:         {train_time:.2f} seconds")
print(f"testing (prediction) time: {test_time:.2f} seconds")
print(f"total runtime (train + test): {total_time:.2f} seconds")

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['predicted 0', 'predicted 1'],
            yticklabels=['actual 0', 'actual 1'])

plt.title("MLP 'n', 'm', 'diameter (A)', 'chiral angle (deg.)' confusion matrix")
plt.xlabel('predicted label')
plt.ylabel('true label')
plt.savefig('mlp_n_conf')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)


plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle="--")
plt.title(f"MLP ROC curve 'n', 'm, 'diameter (A)', 'chiral angle (deg.)' (AUC = {roc_auc:.3f})")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.savefig('roc_curve_n')
plt.show()